ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
%matplotlib inline

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop('CUST_ID', axis=1, inplace=True)
df.head()

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(bins=30, figsize=(18, 14), color='steelblue', edgecolor='black')
plt.suptitle('Distribution of All Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Credit Card Features')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.3, color='royalblue', edgecolors='none')
plt.xlabel('Balance')
plt.ylabel('Purchases')
plt.title('Balance vs Purchases')
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.3, color='tomato', edgecolors='none')
plt.xlabel('Balance')
plt.ylabel('Cash Advance')
plt.title('Balance vs Cash Advance')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print("Scaled data shape:", X_scaled.shape)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

print("Inertia values:", inertia_values)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), inertia_values, marker='o', color='steelblue', linewidth=2)
plt.title('Elbow Method – Inertia vs Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.xticks(range(1, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='darkorange', linewidth=2)
plt.title('Silhouette Score vs Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(range(2, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_table = pd.DataFrame({
    'K': range(2, 11),
    'Silhouette Score': [round(s, 4) for s in silhouette_scores]
})
print(silhouette_table.to_string(index=False))

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
# K=3 gives the best silhouette score (0.2506) and the elbow curve also
# shows a clear slow-down in improvement after K=3.

final_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
final_kmeans.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_kmeans.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
print(df['Cluster'].value_counts().sort_index())

plt.figure(figsize=(6,4))
df['Cluster'].value_counts().sort_index().plot(kind='bar', color=['steelblue','darkorange','green'], edgecolor='black')
plt.title('Number of Customers per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(pca_components, columns=['PCA1', 'PCA2'])
pca_df['Cluster'] = df['Cluster'].values

plt.figure(figsize=(10, 6))
colors = ['steelblue', 'darkorange', 'green']
for cluster in sorted(pca_df['Cluster'].unique()):
    subset = pca_df[pca_df['Cluster'] == cluster]
    plt.scatter(subset['PCA1'], subset['PCA2'],
                label=f'Cluster {cluster}', alpha=0.4,
                color=colors[cluster], edgecolors='none', s=20)

plt.title('K-Means Clusters Visualized with PCA (2 Components)')
plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%")

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

**1. Why is this an unsupervised learning problem?**

Because the dataset does not contain any predefined target labels or categories. We do not know in advance how many customer groups exist or which group each customer belongs to. K-Means discovers these groups purely from the patterns in the data.

---

**2. Why did we remove the `CUST_ID` column?**

`CUST_ID` is a unique identifier for each customer. It carries no behavioral information and would add meaningless noise to the distance calculations used by K-Means. Including it would cause the algorithm to group customers based on their ID numbers rather than their actual spending behavior.

---

**3. Which columns had missing values?**

Two columns had missing values:
- `MINIMUM_PAYMENTS`: 313 missing values
- `CREDIT_LIMIT`: 1 missing value

---

**4. How did you handle the missing values?**

We used **mean imputation** — replacing each missing value with the average value of that column. This preserves the overall distribution of the data without removing any rows.

---

**5. Why is scaling important before applying K-Means?**

K-Means calculates distances between data points to form clusters. If features have very different scales — for example, `BALANCE` can reach thousands of dollars while `PURCHASES_FREQUENCY` is between 0 and 1 — the algorithm will be dominated by the large-scale features and ignore the small-scale ones. Scaling ensures that all features contribute equally to the distance calculation.

---

**6. Which K value did you choose? Explain your answer.**

We chose **K = 3**.

- **Elbow method**: The inertia curve shows a noticeable slowdown after K=3, meaning adding more clusters beyond 3 gives diminishing returns.
- **Silhouette score**: K=3 produced the highest silhouette score of **0.2506**, which indicates the best cluster separation among all tested values.

---

**7. Based on the cluster summary table, describe each customer segment:**

| Cluster | Description |
|---|---|
| **Cluster 0** | High-balance, low-purchase customers. High BALANCE (~3,989) but very low PURCHASES (~385) and low PRC_FULL_PAYMENT (~0.03). These customers carry large balances and rarely pay in full — likely **revolvers** who pay minimum amounts. |
| **Cluster 1** | High-spending, active customers. High PURCHASES (~4,269), high CREDIT_LIMIT, and relatively high PRC_FULL_PAYMENT (~0.30). These are **transactors** who use their cards frequently and pay off balances regularly. |
| **Cluster 2** | Low-activity customers. Low BALANCE (~800), low PURCHASES (~506), and moderate payment behavior. These customers have **minimal engagement** with their credit cards — the majority of customers (6,119) fall here. |

---

**8. Which cluster may represent high-value customers?**

**Cluster 1** represents high-value customers. They have the highest purchase amounts (~4,269), the highest payments (~4,151), and a relatively high rate of full payment (0.30). These customers are active, responsible spenders who generate significant transaction revenue for the company.

---

**9. Which cluster may represent customers who rely more on cash advance?**

**Cluster 0** is most likely the cash-advance-reliant segment. These customers have high balances but very low purchase activity, which suggests they access funds through cash advances rather than regular purchases. Their extremely low full-payment rate (0.03) is also consistent with customers who borrow frequently and carry debt.

---

**10. How can a company use these clusters for marketing strategy?**

- **Cluster 0 (Revolvers)**: Offer balance transfer deals, debt consolidation products, or lower interest rate promotions to retain them and help them manage their debt.
- **Cluster 1 (Transactors)**: Target with premium rewards programs, cashback offers, and travel benefits to increase loyalty among these high-value customers.
- **Cluster 2 (Inactive/Low-use)**: Send engagement campaigns, spending incentives, or introductory offers to activate these customers and increase their card usage.
